# CQR Score Smoke Test

This notebook checks the reviewer-requested CQR-score experiment path. It runs both nonnegative transformations described in the paper: capped scores and shifted scores.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "utility").exists():
    raise RuntimeError(f"Run this notebook from the repository root, not {repo_root}")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")

In [ ]:
from utility.exps import run_cqr_synthetic_experiment

methods = ["TSCP_R", "Unscaled", "TSCP_GWC"]

result = run_cqr_synthetic_experiment(
    dim_list=[3],
    sample_list=[30],
    alpha_list=[0.1],
    noise_type="Gaussian",
    noises_list=[[1, 5, 10]],
    trials=100,
    methods=methods,
    score_transform=["capped", "shifted"],
    shift_constant=20.0,
    base_interval_alpha=0.9,
    n_train=int(0.8 * (2000)), n_test=(2000) - int(0.8 * (2000)),
    n_features=6,
    n_informative=6,
    oracle_n_samples=200,
    quantile_model_params={
        "n_estimators": 50,
        "max_depth": 2,
        "learning_rate": 0.08,
        "min_samples_leaf": 5,
    },
)

result.summary_results

In [ ]:
summary = result.summary_results
coord_summary = result.coordinate_summary_results

assert not summary.empty
assert not coord_summary.empty
assert set(summary["method"]) == set(methods)
assert set(summary["score_type"]) == {"cqr"}
assert set(summary["score_transform"]) == {"capped", "shifted"}
assert set(summary["base_interval_alpha"]) == {0.9}
assert set(coord_summary["coordinate"]) == {1, 2, 3}
assert (summary["test_coverage" if "test_coverage" in summary else "test_coverage_avg"] >= 0).all()
assert (coord_summary["coordinate_length_avg"] >= 0).all()

coord_summary.sort_values(["score_transform", "method", "coordinate"])

In [ ]:
coord_summary.pivot_table(
    index=["score_transform", "coordinate"],
    columns="method",
    values="coordinate_length_avg",
)

For capped scores, `coordinate_adjustment_avg` should be nonnegative. For shifted scores, it may be negative because the final adjustment subtracts the shift constant, which corresponds to shrinking the original quantile interval on that coordinate.